# jfinance クイックスタート

[jfinance](https://github.com/sgawa/jfinance) は、金融庁の電子開示システム **EDINET** に
提出された企業開示データを読むためのライブラリです。

利用登録も API キーも要りません。API は yfinance に合わせてあるので、yfinance のコードが
ほぼそのまま動きます。収録は**2016 年度以降**の有価証券報告書・半期報告書・四半期報告書と
大量保有報告書です。

ドキュメント: <https://jfnc.org/ja/>

## インストール

In [ ]:
!pip install -q jfinance

In [ ]:
import jfinance as jf
import pandas as pd

# 長い表の途中を省かせない
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.0f}")

jf.__version__

## 会社を指定する

指定できるのは証券コード（`7203.T`）・EDINET コード（`E02144`）・ISIN（`JP3633400001`）・
EDINET のファンドコード（`G02925`）です。どれでも同じ会社に届きます。

In [ ]:
t = jf.Ticker("7203.T")

t.info["shortName"], t.info["edinetCode"], t.info["isin"]

### 会社の概要

In [ ]:
pd.Series({k: t.info[k] for k in (
    "shortName", "sector", "industry", "marketSegment", "listingStatus",
    "fiscalYearEnd", "fullTimeEmployees", "sharesOutstanding",
    "capitalMillionJpy", "returnOnEquity", "equityRatio", "trailingEps",
)}, name=t.info["symbol"]).to_frame()

## 財務諸表

EDINET には、財務諸表が**提出されたそのままの形**で入っています。科目も、並び順も、
階層もそのままです。`get_statements()` はそれを返します。まずこちらから見ます。
yfinance と同じ形の表は、この節の後ろにあります。

### 書類に入っている表

有価証券報告書 1 通には、財務三表だけでなく数十の表が入っています。`Role` は
EDINET タクソノミでの表の名前です。

In [ ]:
s = t.get_statements()

print(s.shape[0], "行 /", s["Role"].nunique(), "表")
s["Role"].value_counts().head(12)

### 表を 1 つ、提出どおりに

下のヘルパーは表を 1 つ取り出し、`Depth`（階層の深さ）でインデントします。
これで、印刷された財務諸表とほぼ同じ見た目になります。

表の名前は会計基準で変わります。トヨタは IFRS なので損益計算書は
`Consolidated statement of profit or loss (IFRS)` ですが、日本基準の会社では
`Consolidated statement of income` です。名前は上の一覧で探してください。

In [ ]:
def statement(s, keyword, periods=3):
    """名前に keyword を含む表を 1 つ取り出し、階層でインデントして返す。

    金額の無い行は、その表の見出しです。"""
    roles = [r for r in s["Role"].unique()
             if keyword.lower() in r.lower() and not r.startswith("Notes")]
    if not roles:
        raise KeyError(f"{keyword!r} に当たる表が無い")
    d = s[s["Role"] == roles[0]]
    years = [c for c in d.columns if isinstance(c, int)][-periods:]
    out = d[years].apply(lambda col: col.map(lambda v: "" if pd.isna(v) else f"{v:,.0f}"))
    out.index = ["    " * int(n) + str(lab) for n, lab in zip(d["Depth"], d["Label"])]
    out.index.name = roles[0]
    return out


statement(s, "profit or loss")

### 連結財政状態計算書（貸借対照表）

In [ ]:
statement(s, "financial position")

### 連結キャッシュ・フロー計算書

In [ ]:
statement(s, "cash flows")

### 注記

注記も表として入っています。セグメント情報、有形固定資産、棚卸資産などが、
`Notes -` で始まる `Role` として並びます。

In [ ]:
pd.Series([r for r in s["Role"].unique() if r.startswith("Notes -")], name="Notes")

### 同じ数字を、yfinance と同じ形で

`financials`・`balance_sheet`・`cashflow` は、提出された科目を yfinance の科目名に
まとめ直したものです。yfinance 向けに書いたコードがそのまま動きます。行が科目、
列が期末日（新しい順）、金額は円です。

yfinance との違いは 2 つあります。

- 直近 4 期だけでなく、**収録しているすべての年度**を返します。
- EDINET に開示のある科目だけが並びます。Yahoo 独自の概念（`Normalized EBITDA`・
  `Tax Effect Of Unusual Items` など）は日本の開示に出典が無いので、**行そのものが
  出ません**。これは Yahoo にデータが無い会社と同じ挙動です。並び順は yfinance のままです。

In [ ]:
t.financials

#### 貸借対照表

In [ ]:
t.balance_sheet

#### キャッシュ・フロー計算書

In [ ]:
t.cashflow

### 期間の指定

四半期報告書は 2024 年 4 月に廃止され半期報告書に移りました。そのため四半期の数字は
2023 年度までです。

In [ ]:
for freq in ("yearly", "semiannual", "quarterly"):
    df = t.get_income_stmt(freq=freq)
    print(f"{freq:12} {df.shape[0]:3} 行 x {df.shape[1]:2} 期   "
          f"{df.columns.min().date()} .. {df.columns.max().date()}")

### Yahoo に同名の科目が無い項目

経常利益、1 株当たり純資産、従業員数、それに銀行業・保険業に固有の科目などです。
その会社が開示している項目だけが返るので、製造業に銀行の科目は出ません。

In [ ]:
t.get_jp_financials()

### 連結区分と訂正の反映

既定では、連結があれば連結、そして訂正報告書を反映した値を返します。

In [ ]:
jf.set_financials_basis(consolidation="standalone", version="as_filed")
standalone = jf.Ticker("7203.T").financials

jf.set_financials_basis()          # 既定へ戻す
consolidated = jf.Ticker("7203.T").financials

pd.DataFrame({"連結・訂正反映": consolidated.loc["Total Revenue"],
              "単体・提出時": standalone.loc["Total Revenue"]})

## 株主

日本の開示では、株主の情報が複数の書類に分かれています。jfinance はそれらを混ぜずに、
書類ごとに分けて返します。

### 大株主の状況（有価証券報告書）

In [ ]:
t.major_shareholders

### 所有者別状況

In [ ]:
t.major_holders

### 大量保有報告書（5% ルール）

会社ではなく、保有者が提出する書類です。機関投資家だけでなく、事業会社や個人も含みます。

In [ ]:
h = jf.Ticker("4755.T")            # 楽天グループ
h.large_holders

保有者が報告した売買の明細です。取得・処分の別と単価まで入っています。

In [ ]:
h.large_holder_transactions

## 役員と報酬

役員は年 1 回、有価証券報告書で開示されます。個別の報酬額は連結報酬等が 1 億円以上の
場合だけ開示されるため、ほとんどの役員は `officer_compensation` に出てきません。

### 役員の状況

In [ ]:
t.officers

### 個別報酬（全年度）

In [ ]:
t.officer_compensation

### 役員区分ごとの報酬と、監査報酬

In [ ]:
display(t.officer_remuneration.head(8))
t.audit_fees.head(6)

## セグメント・従業員・提出書類

### セグメント別の売上高

In [ ]:
seg = t.segments.query("Metric == 'seg_revenue'")
seg.pivot_table(index="Segment Label", columns="Fiscal Year",
                values="Value", aggfunc="first").tail(10)

### 従業員の状況

In [ ]:
t.employees

### 提出書類

In [ ]:
# 種類コード 120 = 有価証券報告書
t.get_filings(types=["120"], limit=10)[
    ["Filing Date", "Title", "Fiscal Year", "Is Correction", "Document ID"]]

### 温室効果ガス排出量

XBRL でのタグ付けは 2024 年度の有価証券報告書からなので、まだ空の会社が多くあります。
これは開示された値そのもので、第三者による ESG スコアではありません。

In [ ]:
jf.Ticker('9432.T').emissions      # NTT

## 会社を探す

### 検索

In [ ]:
pd.DataFrame(jf.Search("トヨタ").quotes)[["symbol", "shortname", "quoteType"]]

### 業種分類

In [ ]:
display(jf.JpSector.all())
jf.JpSector("automobiles-transportation-equipment").top_companies.head(10)

### スクリーナー

開示された数字で絞り込みます。書き方は yfinance の `screen` と同じです。

株価から導く項目（時価総額・PER・株価）は EDINET に無いので、指定できません。

In [ ]:
res = jf.edinet_screen(
    jf.EdinetQuery("and", [
        jf.EdinetQuery("gt", ["roe", 0.15]),
        jf.EdinetQuery("gt", ["equityRatio", 0.5]),
    ]),
    sortField="revenue",
    size=20,
)
print(res["total"], "社が該当")
pd.DataFrame(res["quotes"])[
    ["symbol", "shortName", "sector", "roe", "equityRatio", "revenue"]]

#### 指定できる項目

In [ ]:
q = jf.EdinetQuery("gt", ["roe", 0.2])
for group, fields in q.valid_fields.items():
    print(f"{group:12} {len(fields):3}  {', '.join(list(fields)[:6])}")

## 投資信託

決算日ごとに 25 項目。EDINET のファンドコード（`G` で始まる）で指定します。
上場している投信は証券コードでも指定できます。

In [ ]:
jf.Ticker("1306.T").get_fund_financials()

## 日付を指定して提出書類を見る

全社分をまとめて見られます。

In [ ]:
jf.FilingCalendar("2026-06-25").get_filings(types=["120"], limit=10)[
    ["Filing Date", "Title", "Document ID"]]

## 日本語で取得する

社名と業種は、既定では英語で返ります。

事業の内容、役員や株主の氏名は、どちらの設定でも日本語のままです。EDINET に英語の
原文が無いためです。

In [ ]:
jf.config.locale.lang = "ja-JP"

t = jf.Ticker("7203.T")
print(t.info["shortName"], "/", t.info["sector"], "/", t.info["industry"])

## 扱わないもの

株価、配当履歴、株式分割、オプション、アナリスト予想、ニュース、決算発表予定、
ESG スコアは EDINET にありません。これらの yfinance の属性は**定義していない**ので、
空の値が返るのではなく `AttributeError` になります。

In [ ]:
for name in ("history", "dividends", "splits", "news",
             "recommendations", "earnings_dates", "sustainability"):
    print(f"{name:16} {'present' if hasattr(t, name) else 'not defined'}")

## お読みください

**EDINET の閲覧期間が満了した書類に対する訂正報告書は取得できないため、反映されない
場合があります。**古い年度ほど、訂正前の値が残っている可能性があります。

In [ ]:
print(jf.NOTICE_CORRECTIONS)

---

出典と利用条件:

```text
出典：EDINET閲覧（提出）サイト（https://disclosure2.edinet-fsa.go.jp/）、
      PDL1.0（https://www.digital.go.jp/resources/open_data/public_data_license_v1.0）
EDINET閲覧（提出）サイト（https://disclosure2.edinet-fsa.go.jp/）をもとに jfinance 作成
```

同じ表示は、すべての応答のヘッダ `X-JF-Notice` にも入っています。